# ArborGrid · 04 Consultas do MVP (Requisito 07)
Cada consulta responde a uma pergunta do MVP. "Hoje" = o dia de referência mais recente da Ouro.
No relatório: pergunta, SQL, print do resultado e 1–2 frases de interpretação.

In [0]:
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


### P1 · "A próxima tempestade vai atingir o meu bairro? Onde as árvores podem cair sobre a fiação nas próximas 48h?"

In [0]:
WITH ref AS (SELECT MAX(sk_data) AS sk FROM workspace.gs.ouro_fato_risco_arboreo)
SELECT
  r.regiao,
  MAX(f.rajada_max_48h)                                              AS rajada_max_48h_kmh,
  COUNT(*)                                                           AS arvores_monitoradas,
  SUM(CASE WHEN a.triagem_arvore = 'CRITICO' THEN 1 ELSE 0 END)      AS arvores_criticas,
  SUM(CASE WHEN f.priority_tier = 'URGENTE' THEN 1 ELSE 0 END)       AS urgentes,
  SUM(CASE WHEN f.priority_tier = 'ALTA' THEN 1 ELSE 0 END)          AS altas,
  ROUND(AVG(f.composite_risk_score), 1)                              AS score_medio,
  CASE WHEN SUM(CASE WHEN f.priority_tier = 'URGENTE' THEN 1 ELSE 0 END) > 0 THEN 'ALERTA'
       WHEN SUM(CASE WHEN f.priority_tier = 'ALTA' THEN 1 ELSE 0 END) > 0    THEN 'ATENÇÃO'
       ELSE 'NORMAL' END                                             AS situacao
FROM workspace.gs.ouro_fato_risco_arboreo f
JOIN ref ON f.sk_data = ref.sk
JOIN workspace.gs.ouro_dim_arvore a ON a.geosampa_id = f.geosampa_id
JOIN workspace.gs.ouro_dim_regiao r ON r.region_key = f.region_key
GROUP BY r.regiao
ORDER BY score_medio DESC;

regiao,rajada_max_48h_kmh,arvores_monitoradas,arvores_criticas,urgentes,altas,score_medio,situacao
Alto da Mooca (Zona Leste),32.8,50,22,0,22,58.6,ATENÇÃO
Vila Mariana (Zona Sul),35.6,53,28,0,28,58.3,ATENÇÃO
Pinheiros (Zona Oeste),35.6,85,47,0,47,58.1,ATENÇÃO
Avenida Paulista (controle: rede subterrânea),35.6,74,9,0,9,30.2,ATENÇÃO


### P2 · "Quando o risco chega, e quanto tempo eu tenho para me preparar?"

In [0]:
WITH ref AS (
  SELECT to_date(CAST(MAX(sk_data) AS STRING), 'yyyyMMdd') AS dia
  FROM workspace.gs.ouro_fato_risco_arboreo
),
janela AS (
  SELECT c.*, ref.dia
  FROM workspace.gs.ouro_fato_clima_hora c CROSS JOIN ref
  WHERE c.data_hora >= CAST(ref.dia AS TIMESTAMP)
    AND c.data_hora <  CAST(date_add(ref.dia, 2) AS TIMESTAMP)
)
SELECT
  r.regiao,
  MAX(j.rajada_kmh)                                                  AS rajada_max_kmh,
  max_by(j.data_hora, j.rajada_kmh)                                  AS hora_do_pico,
  CAST((unix_timestamp(max_by(j.data_hora, j.rajada_kmh))
        - unix_timestamp(CAST(MAX(j.dia) AS TIMESTAMP))) / 3600 AS INT) AS horas_de_antecedencia,
  SUM(CASE WHEN j.rajada_kmh >= 30 THEN 1 ELSE 0 END)                AS horas_vento_moderado,
  SUM(CASE WHEN j.vento_forte THEN 1 ELSE 0 END)                     AS horas_vento_forte,
  ROUND(SUM(j.chuva_mm), 1)                                          AS chuva_48h_mm
FROM janela j
JOIN workspace.gs.ouro_dim_regiao r ON r.region_key = j.region_key
GROUP BY r.regiao
ORDER BY rajada_max_kmh DESC;

regiao,rajada_max_kmh,hora_do_pico,horas_de_antecedencia,horas_vento_moderado,horas_vento_forte,chuva_48h_mm
Pinheiros (Zona Oeste),35.6,2026-11-17T17:00:00.000-03:00,17,3,0,4.3
Vila Mariana (Zona Sul),35.6,2026-11-17T17:00:00.000-03:00,17,3,0,4.3
Avenida Paulista (controle: rede subterrânea),35.6,2026-11-17T17:00:00.000-03:00,17,3,0,4.3
Alto da Mooca (Zona Leste),32.8,2026-11-17T17:00:00.000-03:00,17,6,0,4.3


### P3 · "Quantas árvores perigosas o ArborGrid encontrou sozinho, antes de qualquer morador denunciar?"
Árvores CRÍTICAS identificadas pelo software sem nenhuma denúncia num raio de 150 m, e o atraso médio
das denúncias que chegaram (falta de sinal no celular).

In [0]:
WITH criticas AS (
  SELECT * FROM workspace.gs.ouro_dim_arvore WHERE triagem_arvore = 'CRITICO'
),
denuncias_proximas AS (
  SELECT a.geosampa_id, a.region_key, COUNT(d.id_denuncia) AS denuncias_ate_150m
  FROM criticas a
  LEFT JOIN workspace.gs.ouro_fato_denuncia d
    ON 111320 * sqrt(pow(a.latitude - d.lat_aprox, 2)
                   + pow((a.longitude - d.lon_aprox) * cos(radians(a.latitude)), 2)) <= 150
  GROUP BY a.geosampa_id, a.region_key
),
por_regiao_denuncias AS (
  SELECT region_key, COUNT(*) AS denuncias_recebidas, ROUND(AVG(atraso_envio_min), 0) AS atraso_medio_envio_min
  FROM workspace.gs.ouro_fato_denuncia
  GROUP BY region_key
)
SELECT
  r.regiao,
  COUNT(*)                                                           AS criticas_identificadas_pelo_software,
  SUM(CASE WHEN p.denuncias_ate_150m = 0 THEN 1 ELSE 0 END)          AS criticas_sem_nenhuma_denuncia,
  ROUND(100.0 * AVG(CASE WHEN p.denuncias_ate_150m = 0 THEN 1 ELSE 0 END), 1) AS pct_sem_denuncia,
  MAX(COALESCE(d.denuncias_recebidas, 0))                            AS denuncias_recebidas_na_regiao,
  MAX(d.atraso_medio_envio_min)                                      AS atraso_medio_envio_min
FROM denuncias_proximas p
JOIN workspace.gs.ouro_dim_regiao r ON r.region_key = p.region_key
LEFT JOIN por_regiao_denuncias d ON d.region_key = p.region_key
GROUP BY r.regiao
ORDER BY criticas_identificadas_pelo_software DESC;

regiao,criticas_identificadas_pelo_software,criticas_sem_nenhuma_denuncia,pct_sem_denuncia,denuncias_recebidas_na_regiao,atraso_medio_envio_min
Pinheiros (Zona Oeste),47,33,70.2,13,55.0
Vila Mariana (Zona Sul),28,21,75.0,12,140.0
Alto da Mooca (Zona Leste),22,18,81.8,13,143.0
Avenida Paulista (controle: rede subterrânea),9,2,22.2,24,30.0


### P4 · "Onde a Defesa Civil e a Enel precisam agir primeiro?" (top 10 árvores)

In [0]:
WITH ref AS (SELECT MAX(sk_data) AS sk FROM workspace.gs.ouro_fato_risco_arboreo)
SELECT
  f.geosampa_id,
  a.distrito,
  r.regiao,
  f.composite_risk_score,
  f.priority_tier,
  a.triagem_arvore,
  a.contato_direto_px_max > 0                                        AS contato_galho_fio,
  COALESCE(CAST(a.ultima_vistoria_sisgau AS STRING), 'nunca vistoriada') AS ultima_vistoria_oficial,
  f.rajada_max_48h                                                   AS rajada_max_48h_kmh,
  f.recommended_action
FROM workspace.gs.ouro_fato_risco_arboreo f
JOIN ref ON f.sk_data = ref.sk
JOIN workspace.gs.ouro_dim_arvore a ON a.geosampa_id = f.geosampa_id
JOIN workspace.gs.ouro_dim_regiao r ON r.region_key = f.region_key
ORDER BY f.composite_risk_score DESC, a.contato_direto_px_max DESC, a.wind_buffer_px_max DESC
LIMIT 10;

geosampa_id,distrito,regiao,composite_risk_score,priority_tier,triagem_arvore,contato_galho_fio,ultima_vistoria_oficial,rajada_max_48h_kmh,recommended_action
635210,PINHEIROS,Pinheiros (Zona Oeste),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
597846,PINHEIROS,Pinheiros (Zona Oeste),80.0,ALTA,CRITICO,true,2013-04-26,35.6,Agendar poda prioritária em até 7 dias
597849,PINHEIROS,Pinheiros (Zona Oeste),80.0,ALTA,CRITICO,true,2013-04-26,35.6,Agendar poda prioritária em até 7 dias
634921,PINHEIROS,Pinheiros (Zona Oeste),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
238790,VILA MARIANA,Vila Mariana (Zona Sul),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
634136,PINHEIROS,Pinheiros (Zona Oeste),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
240683,VILA MARIANA,Vila Mariana (Zona Sul),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
7163,BELA VISTA,Avenida Paulista (controle: rede subterrânea),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
7457,BELA VISTA,Avenida Paulista (controle: rede subterrânea),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias
236976,VILA MARIANA,Vila Mariana (Zona Sul),80.0,ALTA,CRITICO,true,nunca vistoriada,35.6,Agendar poda prioritária em até 7 dias


### P5 · IMPACTO · "Sem prevenção, quantas árvores críticas ficariam expostas na próxima ventania, e quantas podas precisam ser feitas antes?"

In [0]:
SELECT
  d.data,
  d.dia_semana,
  MAX(f.rajada_max_48h)                                              AS rajada_max_48h_kmh,
  COUNT(*)                                                           AS arvores_monitoradas,
  SUM(CASE WHEN a.triagem_arvore = 'CRITICO' THEN 1 ELSE 0 END)      AS arvores_criticas_expostas,
  SUM(CASE WHEN f.priority_tier IN ('URGENTE', 'ALTA') THEN 1 ELSE 0 END) AS podas_prioritarias,
  SUM(CASE WHEN f.priority_tier = 'URGENTE' THEN 1 ELSE 0 END)       AS podas_emergenciais,
  ROUND(100.0 * AVG(CASE WHEN a.ultima_vistoria_sisgau IS NULL THEN 1 ELSE 0 END), 1) AS pct_nunca_vistoriadas
FROM workspace.gs.ouro_fato_risco_arboreo f
JOIN workspace.gs.ouro_dim_data d   ON d.sk_data = f.sk_data
JOIN workspace.gs.ouro_dim_arvore a ON a.geosampa_id = f.geosampa_id
GROUP BY d.data, d.dia_semana
ORDER BY d.data;

data,dia_semana,rajada_max_48h_kmh,arvores_monitoradas,arvores_criticas_expostas,podas_prioritarias,podas_emergenciais,pct_nunca_vistoriadas
2026-11-10,terça,36.7,85,47,47,0,63.5
2026-11-11,quarta,40.0,135,69,69,0,77.0
2026-11-12,quinta,40.0,188,97,97,0,83.5
2026-11-13,sexta,35.3,262,106,106,0,88.2
2026-11-14,sábado,38.2,262,106,106,0,88.2
2026-11-15,domingo,39.2,262,106,106,0,88.2
2026-11-16,segunda,39.2,262,106,106,0,88.2
2026-11-17,terça,35.6,262,106,106,0,88.2
